In [2]:
import pandas as pd

df = pd.read_csv("../data/1976-2024-house.tab")
print(df.shape)
print(df.head(2))
# print(df.dtypes)

C:\Users\Таня\AppData\Local\Temp\ipykernel_16556\2909403207.py:3: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/1976-2024-house.tab")


(33805, 20)
   year    state state_po  state_fips  state_cen  state_ic    office  \
0  1976  ALABAMA       AL           1         63        41  US HOUSE   
1  1976  ALABAMA       AL           1         63        41  US HOUSE   

   district stage runoff  special       candidate       party  writein   mode  \
0         1   GEN  False    False  BILL DAVENPORT    DEMOCRAT    False  TOTAL   
1         1   GEN  False    False    JACK EDWARDS  REPUBLICAN    False  TOTAL   

   candidatevotes  totalvotes  unofficial   version  fusion_ticket  
0           58906      157170       False  20250910          False  
1           98257      157170       False  20250910          False  


In [42]:
# Alabama is roughly 7 times bigger than Alaska population-wise and that's why there are 7x districts
print(df[(df['state_po'] == 'AL') & (df['year'] == 2024) & (df['stage'] == 'GEN')]['district'].unique())
print(df[(df['state_po'] == 'AK') & (df['year'] == 2024) & (df['stage'] == 'GEN')]['district'].unique())

[1 2 3 4 5 6 7]
[0]


### Drop useless columns ###

In [43]:
for col in df.columns:
    print(f"\n--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts().head(10))

# first three are just a different way to encode state_po
# fusion_ticket is a labeling quirk (mainly NY), doesn't change who won
useless_cols = ["state_fips", "state_cen", "state_ic", "fusion_ticket"]

# these cols have just one value - drop them
cols_with_one_value = ["office", "mode", "version"]

# these cols have two values, filter out the minor category, then drop the col
# stage == PRI → intra-party contests (e.g. two Democrats fighting with each other)
# special == TRUE → special elections (vacancy mid-term), different dynamics than regular elections
# runoff == TRUE → a second round after no one hit a threshold, mostly a Southern thing
# unofficial == TRUE → uncertified results
# writein == TRUE → filter out, write-in candidates never win and distort vote totals
filter_then_drop = ["stage", "special", "runoff", "unofficial", "writein"]


--- year (25 unique values) ---
1996    1660
2000    1608
1992    1524
2010    1500
2020    1467
2016    1440
2002    1433
2012    1427
2008    1381
2006    1374
Name: year, dtype: int64

--- state (51 unique values) ---
NEW YORK        4137
CALIFORNIA      3447
TEXAS           2089
MICHIGAN        1641
NEW JERSEY      1332
FLORIDA         1290
ILLINOIS        1261
PENNSYLVANIA    1232
OHIO            1192
VIRGINIA         878
Name: state, dtype: int64

--- state_po (51 unique values) ---
NY    4137
CA    3447
TX    2089
MI    1641
NJ    1332
FL    1290
IL    1261
PA    1232
OH    1192
VA     878
Name: state_po, dtype: int64

--- state_fips (51 unique values) ---
36    4137
6     3447
48    2089
26    1641
34    1332
12    1290
17    1261
42    1232
39    1192
51     878
Name: state_fips, dtype: int64

--- state_cen (51 unique values) ---
21    4137
93    3447
74    2089
34    1641
22    1332
59    1290
33    1261
23    1232
31    1192
54     878
Name: state_cen, dtype: int64

--- sta

In [48]:
if "stage" in df.columns:
    df = df[df["stage"] == "GEN"].copy()

for col in filter_then_drop[1:]:
    if col in df.columns:
        df = df[df[col] == False]

cols_to_drop = [c for c in useless_cols + cols_with_one_value + filter_then_drop if c in df.columns]
df = df.drop(columns=cols_to_drop).reset_index(drop=True).copy()

print(df.shape)
print(df.columns.tolist())

(23074, 8)
['year', 'state', 'state_po', 'district', 'candidate', 'party', 'candidatevotes', 'totalvotes']
